# Split-MODF: Aufbau von `compute_split_modf_main_grid`

Diese Darstellung zeigt den Ablauf der Funktion [`compute_split_modf_main_grid`](/workspaces/ToOp/packages/dc_solver_pkg/src/toop_engine_dc_solver/jax/split_modf_main_grid.py:644).

Die Funktion behandelt **Islanding-Fälle**: Wenn Leitungsausfälle das Netz in mehrere Komponenten trennen, wird nur die Komponente berechnet, welche den ursprünglichen Slack-Bus enthält.

## 1. Gesamtablauf

```mermaid
flowchart TD
    A[compute_split_modf_main_grid] --> B[Statische Eingaben
PTDF, from_node, to_node]
    A --> C[Dynamische Eingaben
nodal_injections, n_0_flow]
    A --> D[Contingencies
outages, branches_monitored, slack_bus]

    B --> E[main_grid_reachable_masks_parallel]
    D --> E
    E --> F[Entfernung der ausgefallenen Leitungen]
    F --> G[main_grid_component_labels
per Contingency-Fall]
    G --> H[main_bus-Maske
Busse der Slack-Komponente]

    B --> I[compute_split_modf_main_grid_from_mask]
    C --> I
    D --> I
    H --> I
    I --> J[jax.vmap über alle Contingency-Fälle]
    J --> K[_compute_split_modf_main_grid_case_from_mask]

    K --> L{Ergebnis}
    L --> M[flows
time x case x monitored]
    L --> N[main_bus
case x bus]
    L --> O[main_monitored
case x monitored]
    L --> P[main_imbalance
time x case]
    L --> Q[rank_loss
case]
```

Die Funktion besitzt zwei logisch getrennte Stufen:

1. **Topologie-Stufe:** Ermittelt, welche Bussen nach jedem Ausfall zur Slack-Komponente gehören.
2. **Elektrische Stufe:** Berechnet die Flüsse in dieser Komponente mit Split-MODF.

### Funktionen im Gesamtablauf und ihre Aufgaben

| Funktion | Aufgabe |
|---|---|
| **`compute_split_modf_main_grid(...)`** | Einstiegspunkt und orchestrierende Funktion. Sie ermittelt zuerst die Slack-Komponente für alle Contingency-Fälle und reicht die entstandenen Busmasken an den elektrischen Split-MODF-Pfad weiter. |
| **`main_grid_reachable_masks_parallel(...)`** | Verarbeitet einen ganzen Contingency-Batch. Für jeden Ausfallsatz ruft sie die Komponentenbestimmung auf und liefert je Fall eine Maske `main_bus[n_bus]`, deren `True`-Einträge zur Slack-Komponente gehören. |
| **`main_grid_component_labels(...)`** | Entfernt die gültigen Leitungsausfälle aus dem Netzgraphen und bestimmt für einen einzelnen Fall alle Zusammenhangskomponenten. Sie gibt pro Bus eine kanonische Komponentenbezeichnung zurück. |
| **`compute_split_modf_main_grid_from_mask(...)`** | Verarbeitet vorberechnete oder gerade ermittelte `main_bus`-Masken für einen ganzen Contingency-Batch. Sie verwendet `jax.vmap`, um den elektrischen Kern einmal je Fall aufzurufen. |
| **`_compute_split_modf_main_grid_case_from_mask(...)`** | Führt die eigentliche elektrische Split-MODF-Berechnung für einen einzelnen Fall aus. Sie verwirft Einspeisungen außerhalb der Hauptkomponente, gleicht die verbliebene Komponente am Slack-Bus aus und korrigiert die N-0-Flüsse. |
| **`_build_split_denominator_and_projected_numerator(...)`** | Erstellt für einen Fall die MODF-Matrix `D = I - H_OO`, den projektierten monitored-Ausfallnumerator und die Masken für gültige Ausfälle. Pufferwerte werden zu neutralen Einheitsrichtungsgrößen ergänzt. |
| **`_pinv_small(...)`** | Berechnet mit einer kleinen SVD die Moore-Penrose-Inverse des MODF-Denominators. Nur Singularwerte oberhalb von `rank_tol` werden invertiert. Damit werden singuläre Islanding-Richtungen verworfen. |
#### Datenfluss zwischen den Funktionen

```text
compute_split_modf_main_grid
  ├── verwendet: main_grid_reachable_masks_parallel
  │     └── verwendet pro Fall: main_grid_component_labels
  │           └── erzeugt: main_bus[n_cases, n_bus]
  │
  └── verwendet: compute_split_modf_main_grid_from_mask
        └── verwendet über jax.vmap pro Fall:
              _compute_split_modf_main_grid_case_from_mask
              ├── verwendet: _build_split_denominator_and_projected_numerator
              └── verwendet: _pinv_small
                    └── erzeugt die elektrisch nutzbare MODF-Inverse
```

Die Zuständigkeiten sind damit klar getrennt:

- Die `main_grid_*`-Funktionen kümmern sich um die Topologie und Zusammenhangskomponenten.
- `compute_split_modf_main_grid_from_mask` organisiert die Contingency-Batches.
- `_compute_split_modf_main_grid_case_from_mask` kombiniert Topologieinformation und elektrische Daten.
- `_build_split_denominator_and_projected_numerator` und `_pinv_small` bilden die numerischen MODF-Bausteine.

## 2. Topologieanalyse

```mermaid
flowchart TD
    A[one topology:
from_node, to_node] --> B[Fixed-width Batch:
outages n_cases x max_n_outages]
    B --> C[jax.vmap
Haupt-Iteration über Fälle]
    C --> D[main_grid_component_labels
für einen Contingency-Fall]

    D --> E[Ausfallsindizes validieren
Pufferwerte ignorieren]
    E --> F[Ausgefallene Leitungen markieren]
    F --> G[Ungültige Endpunkte verwerfen]
    G --> H[Initiale Buslabels:
jeder Bus ist 0, 1, 2, ...]
    H --> I[Schleife: Nachbarlabels verbinden]
    I --> J[Pointer Jumping:
Labelpfade verkürzen]
    J --> K{Labeländerung?}
    K -->|Ja| I
    K -->|Nein| L[Alle Labels der Komponenten
zeigen auf den kleinsten Busindex]
    L --> M[Vergleich mit Slack-Komponente:
label == label_slack]
    M --> N[main_bus[n_bus]:
boolesche Maske]
```

Die Graphsuche verwendet die vollständige physikalische Verbindungsmatrix. Leitungen, die zwar nicht überwacht oder nicht als Ausfallkandidat relevant sind, können für die Zusammenhangskomponenten trotzdem wichtig sein und dürfen daher nicht aus `from_node` und `to_node` entfernt werden.

## 3. Elektrischer Split-MODF-Kern

```mermaid
flowchart TD
    A[_compute_split_modf_main_grid_case_from_mask] --> B[Einspeisungen außerhalb
der main_bus-Maske auf 0]
    B --> C[main_imbalance
Summe der Hauptnetzeinspeisungen]
    C --> D[Ausgleich am Slack-Bus:
p_main = p_main - main_imbalance]
    D --> E[delta_p
= p_main - nodal_injections]

    E --> F[N-0-Korrektur]
    F --> G[base_mon
= n_0_flow_monitored
+ delta_p x PTDF_monitored]
    F --> H[base_out
= n_0_flow_outage
+ delta_p x PTDF_outage]

    A --> I[_build_split_denominator_and_projected_numerator]
    I --> J[H_OO
Outage-Sensitivitäten]
    J --> K[Denominator D = I - H_OO]
    I --> L[Projektierter Numerator nom]

    K --> M[_pinv_small]
    M --> N[SVD von D.T]
    N --> O[Singulärwerte größer rank_tol
umkehren]
    O --> P[Singuläre Richtungen auf 0 setzen]
    P --> R[Moore-Penrose-Inverse
denom_pinv_t]

    R --> S[modf_mon
= denom_pinv_t x nom.T transponiert]
    G --> T[Gesamtfluss
base_mon + modf_mon x base_out]
    S --> T

    T --> U[Überwachte Leitungsendpunkte prüfen]
    U --> V[main_monitored
beide Endpunkte in main_bus]
    V --> W{Ausgefallene überwachte Leitung?}
    W -->|Ja| X[Fluss exakt auf 0 setzen]
    W -->|Nein| Y[Berechneten Fluss behalten]
    X --> Z{main_monitored?}
    Y --> Z
    Z -->|Nein| AA[NaN statt eines erfundenen Flusses]
    Z -->|Ja| AB[Fluss zurückgeben]

    K --> AC[Rangverlust
= Matrixgröße - numerischer Rang]
```

Beim normalen MODF würde `D` invertiert werden. Bei Islanding ist `D` jedoch singulär. Die Pseudoinverse verwirft nur nicht lösbare Richtungen und behält elektrische Richtungen, die im gemischten N-k-Fall noch identifizierbar sind.

## 4. Interaktion der Batch-Ebenen

```mermaid
flowchart LR
    A[Eingabe-Topologie] -->|once| B[PTDF + von/bis]
    B --> C{Contingency-Batch}
    C -->|Fall 1| D1[Main-Bus-Maske 1]
    C -->|Fall 2| D2[Main-Bus-Maske 2]
    C -->|Fall k| Dk[Main-Bus-Maske k]

    D1 --> E1[Split-MODF 1]
    D2 --> E2[Split-MODF 2]
    Dk --> Ek[Split-MODF k]

    F[alle Zeitstempel] --> E1
    F --> E2
    F --> Ek

    E1 --> G[Ausgabe
time x case x monitored]
    E2 --> G
    Ek --> G

    H[äußerer vmap
verschiedene Topologien] -.-> C
```

Die Datenstruktur folgt der Produktionsausgabe:

- `flows`: `n_timesteps × n_cases × n_monitored`
- `main_bus`: `n_cases × n_bus`
- `main_monitored`: `n_cases × n_monitored`
- `main_imbalance`: `n_timesteps × n_cases`
- `rank_loss`: `n_cases`

Fälle mit unterschiedlicher Ausfallanzahl werden als getrennte feste Blöcke verarbeitet, da JAX für `outages` eine statische Form benötigt.

## 5. Schneller Pfad und Fallback

```mermaid
flowchart TD
    A[Islanding-Ausfall] --> B{Main-Bus-Maske
bereits bekannt?}
    B -->|Ja| C[compute_split_modf_main_grid_from_mask]
    C --> D[Keine Graphsuche
im Ausführungs-Pfad]
    B -->|Nein| E[compute_split_modf_main_grid]
    E --> F[main_grid_reachable_masks_parallel]
    F --> C
    D --> G[Elektrischer Split-MODF-Kern]
    C --> G
```

`compute_split_modf_main_grid` ist der robuste dynamische Fallback. Bekannte Islanding-Fälle, etwa feste Brückenausfälle oder vorberechnete 3-Wicklungs-Transformator-Fälle, sollten direkt `compute_split_modf_main_grid_from_mask` verwenden.

Split grid outage -> Ein Ausfall von einer oder mehreren Leitungen/Branches, durch den das Stromnetz in mehrere elektrisch voneinander getrennte Teilnetze zerfällt.